# 04 — Gold: Dimensões SCD Tipo 2

Implementa SCD Tipo 2 para `DimCustomer` e `DimProduct` no DuckDB.

**Padrão:**
1. **Carga inicial** — `ValidFrom = '1900-01-01'` (pedidos Northwind: 1996–1998)
2. **Incremental** — detectar mudanças → expirar versão atual → inserir nova versão

**Atributos rastreados:**
- `DimCustomer`: `ContactName`, `ContactTitle`, `City`, `Country`
- `DimProduct`: `UnitPrice`, `Discontinued`

**Técnica DuckDB:**
- `UPDATE ... WHERE CustomerID IN (subquery de mudanças)` para expirar
- `INSERT INTO ... SELECT ... WHERE CustomerID IN (subquery)` para nova versão
- `IS DISTINCT FROM` para comparação NULL-safe (DuckDB suporta nativamente)

In [1]:
import sys, os
from datetime import date
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# DimCustomer — SCD Tipo 2
# Atributos rastreados: ContactName, ContactTitle, City, Country
# ============================================================

def process_customers_scd2():
    today = str(date.today())

    # PASSO 1: Carga inicial (tabela vazia)
    count = conn.execute("SELECT COUNT(*) FROM gold.DimCustomer").fetchone()[0]
    if count == 0:
        conn.execute(f"""
            INSERT INTO gold.DimCustomer
            SELECT
                CAST(hash(CustomerID || '1900-01-01') % 2147483647 AS INTEGER) AS CustomerSK,
                CustomerID, CompanyName, ContactName, ContactTitle, City, Country,
                DATE '1900-01-01' AS ValidFrom,
                DATE '9999-12-31' AS ValidTo,
                TRUE              AS IsCurrent
            FROM bronze.customers
        """)
        n = conn.execute("SELECT COUNT(*) FROM gold.DimCustomer").fetchone()[0]
        print(f"Carga inicial: {n} clientes (ValidFrom = 1900-01-01)")
        return

    # PASSO 2: Detectar mudanças — capturar IDs ANTES de expirar
    # IS DISTINCT FROM: NULL-safe comparison (suporte nativo no DuckDB)
    changed_ids = [
        r[0] for r in conn.execute("""
            SELECT b.CustomerID
            FROM bronze.customers b
            JOIN gold.DimCustomer s ON b.CustomerID = s.CustomerID AND s.IsCurrent = TRUE
            WHERE b.ContactName  IS DISTINCT FROM s.ContactName
               OR b.ContactTitle IS DISTINCT FROM s.ContactTitle
               OR b.City         IS DISTINCT FROM s.City
               OR b.Country      IS DISTINCT FROM s.Country
        """).fetchall()
    ]

    if changed_ids:
        id_list = ", ".join(f"'{x}'" for x in changed_ids)

        # PASSO 2a: Expirar versão atual
        conn.execute(f"""
            UPDATE gold.DimCustomer
            SET ValidTo   = DATE '{today}' - INTERVAL '1 day',
                IsCurrent = FALSE
            WHERE IsCurrent = TRUE
              AND CustomerID IN ({id_list})
        """)

        # PASSO 2b: Inserir nova versão para exatamente os IDs que mudaram
        conn.execute(f"""
            INSERT INTO gold.DimCustomer
            SELECT
                CAST(hash(CustomerID || '{today}') % 2147483647 AS INTEGER) AS CustomerSK,
                CustomerID, CompanyName, ContactName, ContactTitle, City, Country,
                DATE '{today}' AS ValidFrom,
                DATE '9999-12-31' AS ValidTo,
                TRUE AS IsCurrent
            FROM bronze.customers
            WHERE CustomerID IN ({id_list})
        """)
        print(f"Mudanças: {len(changed_ids)} clientes expirados e nova versão inserida")
    else:
        print("Nenhuma mudança detectada em DimCustomer.")

    # PASSO 3: Clientes novos (não existem em silver)
    new_count = conn.execute("""
        SELECT COUNT(*) FROM bronze.customers
        WHERE CustomerID NOT IN (SELECT CustomerID FROM gold.DimCustomer)
    """).fetchone()[0]

    if new_count > 0:
        conn.execute(f"""
            INSERT INTO gold.DimCustomer
            SELECT
                CAST(hash(CustomerID || '{today}') % 2147483647 AS INTEGER) AS CustomerSK,
                CustomerID, CompanyName, ContactName, ContactTitle, City, Country,
                DATE '{today}' AS ValidFrom,
                DATE '9999-12-31' AS ValidTo,
                TRUE AS IsCurrent
            FROM bronze.customers
            WHERE CustomerID NOT IN (SELECT CustomerID FROM gold.DimCustomer)
        """)
        print(f"Novos clientes: {new_count}")


process_customers_scd2()
total   = conn.execute("SELECT COUNT(*) FROM gold.DimCustomer").fetchone()[0]
current = conn.execute("SELECT COUNT(*) FROM gold.DimCustomer WHERE IsCurrent = TRUE").fetchone()[0]
print(f"DimCustomer: {total} versões totais, {current} ativas (IsCurrent=true)")

Carga inicial: 91 clientes (ValidFrom = 1900-01-01)
DimCustomer: 91 versões totais, 91 ativas (IsCurrent=true)


In [3]:
# ============================================================
# DimProduct — SCD Tipo 2 com denormalização
# Atributos rastreados: UnitPrice, Discontinued
# Denormaliza: CategoryName + SupplierCompany
# ============================================================

def process_products_scd2():
    today = str(date.today())

    # PASSO 1: Carga inicial
    count = conn.execute("SELECT COUNT(*) FROM gold.DimProduct").fetchone()[0]
    if count == 0:
        conn.execute(f"""
            INSERT INTO gold.DimProduct
            SELECT
                CAST(hash(CAST(p.ProductID AS VARCHAR) || '1900-01-01') % 2147483647 AS INTEGER) AS ProductSK,
                p.ProductID,
                p.ProductName,
                c.CategoryName,
                s.CompanyName AS SupplierCompany,
                p.UnitPrice,
                p.QuantityPerUnit,
                p.Discontinued,
                DATE '1900-01-01' AS ValidFrom,
                DATE '9999-12-31' AS ValidTo,
                TRUE              AS IsCurrent
            FROM bronze.products p
            LEFT JOIN bronze.categories c ON p.CategoryID = c.CategoryID
            LEFT JOIN bronze.suppliers  s ON p.SupplierID = s.SupplierID
        """)
        n = conn.execute("SELECT COUNT(*) FROM gold.DimProduct").fetchone()[0]
        print(f"Carga inicial: {n} produtos (ValidFrom = 1900-01-01)")
        return

    # PASSO 2: Detectar mudanças — capturar IDs ANTES de expirar
    changed_ids = [
        r[0] for r in conn.execute("""
            SELECT p.ProductID
            FROM bronze.products p
            JOIN gold.DimProduct s ON p.ProductID = s.ProductID AND s.IsCurrent = TRUE
            WHERE p.UnitPrice    IS DISTINCT FROM s.UnitPrice
               OR p.Discontinued IS DISTINCT FROM s.Discontinued
        """).fetchall()
    ]

    if changed_ids:
        id_list = ", ".join(str(x) for x in changed_ids)

        # PASSO 2a: Expirar versão atual
        conn.execute(f"""
            UPDATE gold.DimProduct
            SET ValidTo   = DATE '{today}' - INTERVAL '1 day',
                IsCurrent = FALSE
            WHERE IsCurrent = TRUE
              AND ProductID IN ({id_list})
        """)

        # PASSO 2b: Inserir nova versão para exatamente os IDs que mudaram
        conn.execute(f"""
            INSERT INTO gold.DimProduct
            SELECT
                CAST(hash(CAST(p.ProductID AS VARCHAR) || '{today}') % 2147483647 AS INTEGER) AS ProductSK,
                p.ProductID,
                p.ProductName,
                c.CategoryName,
                s.CompanyName AS SupplierCompany,
                p.UnitPrice,
                p.QuantityPerUnit,
                p.Discontinued,
                DATE '{today}' AS ValidFrom,
                DATE '9999-12-31' AS ValidTo,
                TRUE AS IsCurrent
            FROM bronze.products p
            LEFT JOIN bronze.categories c ON p.CategoryID = c.CategoryID
            LEFT JOIN bronze.suppliers  s ON p.SupplierID = s.SupplierID
            WHERE p.ProductID IN ({id_list})
        """)
        print(f"Mudanças: {len(changed_ids)} produtos com nova versão")
    else:
        print("Nenhuma mudança em DimProduct.")

    # PASSO 3: Produtos novos
    new_count = conn.execute("""
        SELECT COUNT(*) FROM bronze.products
        WHERE ProductID NOT IN (SELECT ProductID FROM gold.DimProduct)
    """).fetchone()[0]

    if new_count > 0:
        conn.execute(f"""
            INSERT INTO gold.DimProduct
            SELECT
                CAST(hash(CAST(p.ProductID AS VARCHAR) || '{today}') % 2147483647 AS INTEGER) AS ProductSK,
                p.ProductID, p.ProductName,
                c.CategoryName, s.CompanyName AS SupplierCompany,
                p.UnitPrice, p.QuantityPerUnit, p.Discontinued,
                DATE '{today}', DATE '9999-12-31', TRUE
            FROM bronze.products p
            LEFT JOIN bronze.categories c ON p.CategoryID = c.CategoryID
            LEFT JOIN bronze.suppliers  s ON p.SupplierID = s.SupplierID
            WHERE p.ProductID NOT IN (SELECT ProductID FROM gold.DimProduct)
        """)
        print(f"Novos produtos: {new_count}")


process_products_scd2()
total   = conn.execute("SELECT COUNT(*) FROM gold.DimProduct").fetchone()[0]
current = conn.execute("SELECT COUNT(*) FROM gold.DimProduct WHERE IsCurrent = TRUE").fetchone()[0]
print(f"DimProduct: {total} versões totais, {current} ativas (IsCurrent=true)")

Carga inicial: 77 produtos (ValidFrom = 1900-01-01)
DimProduct: 77 versões totais, 77 ativas (IsCurrent=true)


In [4]:
# ============================================================
# LAB: Simular mudança e re-executar (demonstra SCD2 em ação)
# ============================================================
print("=" * 55)
print("LAB: Simulação de mudança de dados em bronze")
print("=" * 55)

# Ver cliente ALFKI antes da mudança
print("\nCliente ALFKI — versão atual:")
print(conn.execute("""
    SELECT CustomerSK, CustomerID, ContactName, City, Country, ValidFrom, ValidTo, IsCurrent
    FROM gold.DimCustomer WHERE CustomerID = 'ALFKI'
""").fetchdf().to_string(index=False))

# Simular mudança: alterar ContactName de ALFKI na bronze
conn.execute("UPDATE bronze.customers SET ContactName = 'Maria Anders (UPDATED)' WHERE CustomerID = 'ALFKI'")
print("\nbronze.customers: ContactName de ALFKI atualizado.")

# Re-executar SCD2
process_customers_scd2()

# Ver ALFKI agora: deve ter 2 versões
print("\nCliente ALFKI — versões SCD2 (esperado: 2 linhas):")
print(conn.execute("""
    SELECT CustomerSK, CustomerID, ContactName, City, ValidFrom, ValidTo, IsCurrent
    FROM gold.DimCustomer WHERE CustomerID = 'ALFKI'
    ORDER BY ValidFrom
""").fetchdf().to_string(index=False))

# Validação: exatamente 1 IsCurrent por CustomerID
dups = conn.execute("""
    SELECT COUNT(*) FROM (
        SELECT CustomerID FROM gold.DimCustomer
        WHERE IsCurrent = TRUE
        GROUP BY CustomerID HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"\nClientes com mais de 1 versão ativa: {dups} (esperado: 0)")

LAB: Simulação de mudança de dados em bronze

Cliente ALFKI — versão atual:
 CustomerSK CustomerID  ContactName   City Country  ValidFrom    ValidTo  IsCurrent
 1558206181      ALFKI Maria Anders Berlin Germany 1900-01-01 9999-12-31       True

bronze.customers: ContactName de ALFKI atualizado.
Mudanças: 1 clientes expirados e nova versão inserida

Cliente ALFKI — versões SCD2 (esperado: 2 linhas):
 CustomerSK CustomerID            ContactName   City  ValidFrom    ValidTo  IsCurrent
 1558206181      ALFKI           Maria Anders Berlin 1900-01-01 2026-03-28      False
 1805990513      ALFKI Maria Anders (UPDATED) Berlin 2026-03-29 9999-12-31       True

Clientes com mais de 1 versão ativa: 0 (esperado: 0)


In [5]:
# ============================================================
# Resumo final
# ============================================================
print("\nResumo SCD2:")
for table, nk in [("gold.DimCustomer", "CustomerID"), ("gold.DimProduct", "ProductID")]:
    total   = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    active  = conn.execute(f"SELECT COUNT(*) FROM {table} WHERE IsCurrent = TRUE").fetchone()[0]
    hist    = total - active
    print(f"  {table}: total={total}, ativas={active}, históricas={hist}")

conn.close()


Resumo SCD2:
  gold.DimCustomer: total=92, ativas=91, históricas=1
  gold.DimProduct: total=77, ativas=77, históricas=0
